**Import libraries and set global options.**

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, average_precision_score)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True,
                     "grid.alpha": .25, "axes.spines.top": False,
                     "axes.spines.right": False})

print("pandas", pd.__version__, "| numpy", np.__version__)

**Load the raw synthetic orders CSV.**

In [ ]:
CANDIDATES = [
    Path(r"D:\projects1\ecommerce-fraud-detection\data\raw\ecommerce_orders_fraud.csv"),
    Path("ecommerce_orders_fraud.csv"),
    Path.home() / "OneDrive" / "Desktop" / "ecommerce_orders_fraud.csv",
    Path.home() / "Desktop" / "ecommerce_orders_fraud.csv",
]
CSV_PATH = next((p for p in CANDIDATES if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "ecommerce_orders_fraud.csv not found. Put it next to this notebook, "
        f"or edit CANDIDATES. Looked in: {[str(p) for p in CANDIDATES]}")

OUT_DIR = CSV_PATH.parent
df = pd.read_csv(CSV_PATH)

print("loaded :", CSV_PATH)
print("shape  :", df.shape)
print("\ndtypes:")
print(df.dtypes.to_string())
print("\nfirst 3 rows:")
display(df.head(3))


**Check fraud rate, class imbalance, and basic dataset stats.**

In [ ]:
n_fraud = int(df["is_fraud"].sum())
print(f"fraud orders : {n_fraud:,} / {len(df):,} = {100 * n_fraud / len(df):.2f}%")
print(f"legit orders : {len(df) - n_fraud:,} = {100 * (1 - n_fraud / len(df)):.2f}%")
print(f"imbalance    : 1 fraud for every {(len(df) - n_fraud) / n_fraud:.0f} legit orders")
print(f"\ncustomers {df.customer_id.nunique():,} | devices {df.device_id.nunique():,} | "
      f"addresses {df.shipping_address_id.nunique():,} | IPs {df.ip_address.nunique():,}")

print("\ncategory mix and average basket:")
display(df.groupby("product_category")
          .agg(orders=("order_id", "size"),
               median_amount=("order_amount", "median"),
               mean_amount=("order_amount", "mean"))
          .sort_values("orders", ascending=False))

**Hold out the fraud label so it can't leak into feature engineering.**

In [ ]:
LABEL_COLS = ["is_fraud", "fraud_pattern"]
labels = df[["order_id"] + LABEL_COLS].set_index("order_id")   # locked until Step 10
df = df.drop(columns=LABEL_COLS)                               # gone from the feature frame


def true_labels(frame=None):
    # Re-join the held-out label by order_id, in the frame's CURRENT row order.
    f = df if frame is None else frame
    return f.order_id.map(labels.is_fraud).values


def true_patterns(frame=None):
    f = df if frame is None else frame
    return f.order_id.map(labels.fraud_pattern).values


print("held out for evaluation only:", LABEL_COLS)
print("columns remaining in the feature frame:", len(df.columns))
print("\npattern types present (context only, never used as a feature):")
print(labels.fraud_pattern.value_counts().to_string())

**Check for duplicates and missing values.**

In [ ]:
print("duplicate whole rows       :", df.duplicated().sum())
print("duplicate order_ids        :", df.order_id.duplicated().sum())

print("\nmissing values per column:")
miss = pd.DataFrame({"n_missing": df.isna().sum(),
                     "pct": (100 * df.isna().mean()).round(2)})
display(miss[miss.n_missing > 0])

not_returned = df.is_returned == 0
print("blank return_reason rows that are simply un-returned orders: "
      f"{df.loc[not_returned, 'return_reason'].isna().sum():,} of "
      f"{df.return_reason.isna().sum():,} -> structural, not dirty data")

**Clean data types, fix dates, and run sanity checks.**

In [ ]:
# 1. dates as real datetimes
for c in ["timestamp", "account_creation_date", "return_timestamp"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")

# 2. structural blanks get an explicit category, not an imputed value
df["return_reason"] = df["return_reason"].fillna("not_returned")

# 3. drop exact duplicates (none expected, but the pipeline should be safe to re-run)
before = len(df)
df = df.drop_duplicates(subset="order_id", keep="first").reset_index(drop=True)
print(f"rows: {before:,} -> {len(df):,}")

# 4. sanity checks that would catch a corrupt export
bad_age = (df.timestamp.dt.normalize() < df.account_creation_date).sum()
bad_ret = (df.return_timestamp < df.timestamp).sum()
print("orders placed before the account existed :", bad_age)
print("returns dated before the order           :", bad_ret)
print("order_amount <= 0                        :", (df.order_amount <= 0).sum())
print("\ndate range:", df.timestamp.min(), "->", df.timestamp.max())
print("dtypes after conversion:")
print(df[["timestamp", "account_creation_date", "return_timestamp"]].dtypes.to_string())

**Create time-based features (hour, weekend, account age, billing mismatch).**

In [ ]:
# ---- time features -------------------------------------------------------
df["hour_of_day"] = df.timestamp.dt.hour
df["day_of_week"] = df.timestamp.dt.dayofweek                 # 0 = Monday
df["day_name"] = df.timestamp.dt.day_name()
df["order_date"] = df.timestamp.dt.normalize()
df["is_weekend"] = (df.day_of_week >= 5).astype(int)
df["is_odd_hour"] = ((df.hour_of_day >= 1) & (df.hour_of_day < 5)).astype(int)   # 01:00-04:59

# ---- account age ---------------------------------------------------------
df["account_age_days"] = (df.order_date - df.account_creation_date).dt.days
df["is_new_account"] = (df.account_age_days <= 3).astype(int)

# ---- billing / shipping mismatch ----------------------------------------
df["billing_mismatch"] = (df.billing_address_id != df.shipping_address_id).astype(int)

print(df[["hour_of_day", "day_of_week", "is_odd_hour", "account_age_days",
          "billing_mismatch"]].describe().T)

**Compute each customer's order count in the last 24 hours (velocity).**

In [ ]:
# ---- orders_last_24h : rolling per-customer velocity ---------------------
# For each order, how many orders did THIS customer place in the preceding 24 hours?
# Done with a per-customer binary search instead of a rolling window so it stays O(n log n).
df = df.sort_values(["customer_id", "timestamp"]).reset_index(drop=True)

ts = df.timestamp.values.astype("datetime64[s]").astype(np.int64)
cust = df.customer_id.values
bounds = np.r_[0, np.flatnonzero(cust[1:] != cust[:-1]) + 1, len(df)]

prior_24h = np.zeros(len(df), dtype=int)
for a, b in zip(bounds[:-1], bounds[1:]):
    t = ts[a:b]
    lo = np.searchsorted(t, t - 24 * 3600, side="left")
    prior_24h[a:b] = np.arange(b - a) - lo

df["orders_last_24h"] = prior_24h

print("orders_last_24h distribution:")
print(df.orders_last_24h.value_counts().sort_index().head(8).to_string())
print(f"\nmax burst by a single customer in 24h: {df.orders_last_24h.max()}")

**Compute per-customer history features (order count, return rate, avg spend).**

In [ ]:
# ---- customer-level history ---------------------------------------------
cust_stats = (df.groupby("customer_id")
                .agg(customer_order_count=("order_id", "size"),
                     customer_return_count=("is_returned", "sum"),
                     customer_total_spend=("order_amount", "sum"),
                     customer_avg_amount=("order_amount", "mean"))
                .reset_index())
cust_stats["customer_return_rate"] = (cust_stats.customer_return_count
                                      / cust_stats.customer_order_count)
df = df.merge(cust_stats, on="customer_id", how="left")

# production-safe twin: only orders that already happened, current row excluded
df = df.sort_values(["customer_id", "timestamp"]).reset_index(drop=True)
df["customer_return_rate_to_date"] = (df.groupby("customer_id").is_returned
                                        .transform(lambda s: s.shift().expanding().mean())
                                        .fillna(0))

print("customers with a >=50% return rate and 5+ orders:",
      int(((cust_stats.customer_return_rate >= .5) &
           (cust_stats.customer_order_count >= 5)).sum()))
display(cust_stats.customer_return_rate.describe().to_frame().T)

**Compute entity-reuse features (shared device/address/IP across accounts).**

In [ ]:
# ---- entity reuse : how many DISTINCT accounts share this thing? ---------
for col, name in [("shipping_address_id", "address_reuse_count"),
                  ("device_id", "device_reuse_count"),
                  ("ip_address", "ip_reuse_count")]:
    df[name] = df[col].map(df.groupby(col).customer_id.nunique())

# the pair is stronger than either alone: same roof AND same phone
pair = df.groupby(["shipping_address_id", "device_id"]).customer_id.nunique()
df["address_device_pair_count"] = pd.MultiIndex.from_frame(
    df[["shipping_address_id", "device_id"]]).map(pair)

print(df[["address_reuse_count", "device_reuse_count", "ip_reuse_count",
          "address_device_pair_count"]].describe().T)
print("\norders on a device shared by 5+ accounts:",
      int((df.device_reuse_count >= 5).sum()))

**Compute order amount relative to its product category (z-score, ratio, log).**

In [ ]:
# ---- amount in context of its category ----------------------------------
grp = df.groupby("product_category").order_amount
df["category_mean_amount"] = grp.transform("mean")
df["category_std_amount"] = grp.transform("std")

# z-score within category: "how many standard deviations above normal for this category"
df["order_amount_deviation"] = ((df.order_amount - df.category_mean_amount)
                                / df.category_std_amount)
# ratio version, easier to talk about out loud ("3x the category average")
df["amount_vs_category_avg"] = df.order_amount / df.category_mean_amount
df["log_order_amount"] = np.log1p(df.order_amount)

print(df.groupby("product_category")
        .agg(mean_amount=("order_amount", "mean"),
             mean_deviation=("order_amount_deviation", "mean"),
             max_deviation=("order_amount_deviation", "max"))
        .sort_values("mean_amount", ascending=False))

**Collect the final list of engineered features.**

In [ ]:
ENGINEERED = ["account_age_days", "orders_last_24h", "customer_return_rate",
              "address_reuse_count", "device_reuse_count", "ip_reuse_count",
              "address_device_pair_count", "is_odd_hour", "hour_of_day",
              "day_of_week", "is_weekend", "billing_mismatch", "is_new_account",
              "order_amount", "log_order_amount", "order_amount_deviation",
              "amount_vs_category_avg", "quantity", "is_returned",
              "customer_order_count", "customer_return_rate_to_date"]

print(f"{len(ENGINEERED)} engineered features")
print("any NaNs left?", df[ENGINEERED].isna().sum().sum())
display(df[ENGINEERED].head(3))

**Compare feature distributions between fraud and normal orders.**

In [ ]:
y_true = true_labels()          # re-joined by order_id; diagnostic use only

cmp_rows = []
for f in ENGINEERED:
    a = df.loc[y_true == 0, f].astype(float)
    b = df.loc[y_true == 1, f].astype(float)
    pooled = np.sqrt((a.var() + b.var()) / 2) or 1e-9
    cmp_rows.append({
        "feature": f,
        "normal_mean": a.mean(), "fraud_mean": b.mean(),
        "normal_median": a.median(), "fraud_median": b.median(),
        "ratio_fraud_to_normal": b.mean() / a.mean() if a.mean() else np.nan,
        "separation": abs(b.mean() - a.mean()) / pooled,
    })

comparison = (pd.DataFrame(cmp_rows)
                .set_index("feature")
                .sort_values("separation", ascending=False))
display(comparison)

**Plot the top separating features for fraud vs normal.**

In [ ]:
top = comparison.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, f in zip(axes.ravel(), top):
    a = df.loc[y_true == 0, f].astype(float)
    b = df.loc[y_true == 1, f].astype(float)
    lo, hi = np.nanpercentile(pd.concat([a, b]), [0, 99])
    bins = np.linspace(lo, hi, 30) if hi > lo else 30
    ax.hist(a, bins=bins, density=True, alpha=.6, label="normal")
    ax.hist(b, bins=bins, density=True, alpha=.6, label="fraud")
    ax.set_title(f, fontsize=10)
    ax.legend(fontsize=8)
fig.suptitle("Top 6 features by separation — fraud vs normal", y=1.02)
plt.tight_layout()
plt.show()

print("Takeaway: the strongest signals are entity reuse (one device/address behind many "
      "accounts),\nhow far the basket sits above its category average, and how new the "
      "account is.\nNone of them separates perfectly on its own — which is why we combine "
      "detectors in Step 9.")

**Apply IQR outlier detection (global, per-category, and on the deviation score).**

In [ ]:
def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


# --- 1. naive global IQR on raw amount -----------------------------------
g_lo, g_hi = iqr_bounds(df.order_amount)
df["iqr_flag_global"] = (df.order_amount > g_hi).astype(int)
print(f"global bounds on order_amount: upper = {g_hi:,.0f}")
print(f"  flagged {df.iqr_flag_global.sum():,} orders ({100 * df.iqr_flag_global.mean():.1f}%)")
print("  what it actually caught, by category:")
print(df[df.iqr_flag_global == 1].product_category.value_counts().head(4).to_string())

# --- 2. per-category IQR on raw amount -----------------------------------
cat_hi = df.groupby("product_category").order_amount.transform(lambda s: iqr_bounds(s)[1])
df["iqr_flag_category"] = (df.order_amount > cat_hi).astype(int)
print(f"\nper-category IQR flagged {df.iqr_flag_category.sum():,} orders "
      f"({100 * df.iqr_flag_category.mean():.1f}%)")

# --- 3. IQR on the category z-score --------------------------------------
d_lo, d_hi = iqr_bounds(df.order_amount_deviation)
df["iqr_flag_deviation"] = (df.order_amount_deviation > d_hi).astype(int)
print(f"deviation IQR upper bound = {d_hi:.2f} sd; flagged "
      f"{df.iqr_flag_deviation.sum():,} orders")

# --- combined ------------------------------------------------------------
df["iqr_flag"] = ((df.iqr_flag_category == 1) | (df.iqr_flag_deviation == 1)).astype(int)
print(f"\nfinal iqr_flag: {df.iqr_flag.sum():,} orders ({100 * df.iqr_flag.mean():.1f}%)")

**Visualize IQR results and score its standalone precision/recall.**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cats = sorted(df.product_category.unique())
# note: set_xticklabels rather than boxplot(labels=...) - that kwarg was renamed in
# matplotlib 3.9 and removed in 3.11, so this form works on every version
axes[0].boxplot([df.loc[df.product_category == c, "order_amount"] for c in cats],
                showfliers=True, flierprops=dict(markersize=2, alpha=.3))
axes[0].set_xticks(range(1, len(cats) + 1))
axes[0].set_xticklabels(cats, rotation=90, fontsize=8)
axes[0].set_yscale("log")
axes[0].set_title("order_amount by category (log scale) — why global IQR fails")

axes[1].hist(df.order_amount_deviation, bins=80, color="#4C78A8")
axes[1].axvline(d_hi, color="crimson", ls="--", label=f"IQR upper = {d_hi:.2f}")
axes[1].set_title("order_amount_deviation (category z-score)")
axes[1].set_yscale("log")
axes[1].legend()
plt.tight_layout()
plt.show()

# how good is this baseline? (label used for measurement only)
print(f"iqr_flag  precision {precision_score(y_true, df.iqr_flag):.3f}"
      f"  recall {recall_score(y_true, df.iqr_flag):.3f}"
      f"  F1 {f1_score(y_true, df.iqr_flag):.3f}")
print("A blunt instrument: it fires on genuine big-ticket purchases too. Useful as one "
      "vote,\nnot as a decision.")

**Train the Isolation Forest model on the engineered features.**

In [ ]:
IF_FEATURES = ["account_age_days", "log_order_amount", "order_amount_deviation",
               "quantity", "orders_last_24h", "customer_return_rate",
               "address_reuse_count", "device_reuse_count", "ip_reuse_count",
               "address_device_pair_count", "is_odd_hour", "hour_of_day",
               "day_of_week", "billing_mismatch", "customer_order_count"]

X = df[IF_FEATURES].astype(float).values
CONTAMINATION = 0.05          # ~500 orders queued for review out of 10,000

iso = IsolationForest(n_estimators=300, contamination=CONTAMINATION,
                      max_samples="auto", random_state=RANDOM_STATE, n_jobs=-1)
iso.fit(X)

# score_samples: lower = more anomalous. Negate so HIGHER = more suspicious.
df["anomaly_score"] = -iso.score_samples(X)
df["isolation_forest_flag"] = (iso.predict(X) == -1).astype(int)

print(f"trained on {len(IF_FEATURES)} features, {len(df):,} rows")
print(f"flagged {df.isolation_forest_flag.sum():,} orders "
      f"({100 * df.isolation_forest_flag.mean():.1f}%)")
print(f"anomaly_score range {df.anomaly_score.min():.3f} to {df.anomaly_score.max():.3f}")

**Visualize anomaly scores and score Isolation Forest's precision/recall.**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df.loc[df.isolation_forest_flag == 0, "anomaly_score"], bins=60,
             alpha=.7, label="not flagged")
axes[0].hist(df.loc[df.isolation_forest_flag == 1, "anomaly_score"], bins=60,
             alpha=.8, label="flagged", color="crimson")
axes[0].set_title("Isolation Forest anomaly score")
axes[0].set_xlabel("higher = more anomalous")
axes[0].legend()

axes[1].scatter(df.account_age_days, df.amount_vs_category_avg, s=6, alpha=.25,
                c=np.where(df.isolation_forest_flag == 1, "crimson", "#8899aa"))
axes[1].set_xscale("symlog")
axes[1].set_yscale("log")
axes[1].set_xlabel("account age (days, symlog)")
axes[1].set_ylabel("amount vs category avg (log)")
axes[1].set_title("What the forest isolates: new accounts + oversized baskets")
plt.tight_layout()
plt.show()

print(f"isolation_forest_flag  precision {precision_score(y_true, df.isolation_forest_flag):.3f}"
      f"  recall {recall_score(y_true, df.isolation_forest_flag):.3f}"
      f"  F1 {f1_score(y_true, df.isolation_forest_flag):.3f}")
print(f"anomaly_score ROC-AUC vs true label: "
      f"{roc_auc_score(y_true, df.anomaly_score):.3f}  "
      f"(0.5 = coin flip, 1.0 = perfect)")

**Cluster orders by shared shipping address + device to find fraud rings.**

In [ ]:
# ---- cluster-level view: one row per (shipping address, device) ----------
clusters = (df.groupby(["shipping_address_id", "device_id"])
              .agg(n_customers=("customer_id", "nunique"),
                   n_orders=("order_id", "size"),
                   total_value=("order_amount", "sum"),
                   median_amount=("order_amount", "median"),
                   first_signup=("account_creation_date", "min"),
                   last_signup=("account_creation_date", "max"),
                   median_account_age=("account_age_days", "median"))
              .reset_index())
clusters["signup_window_days"] = (clusters.last_signup - clusters.first_signup).dt.days

MIN_ACCOUNTS = 3           # a cluster starts at 3 distinct accounts
TIGHT_WINDOW = 14          # all created within two weeks = coordinated

clusters["is_multi_account"] = (clusters.n_customers >= MIN_ACCOUNTS).astype(int)
clusters["is_coordinated"] = ((clusters.n_customers >= MIN_ACCOUNTS) &
                              (clusters.signup_window_days <= TIGHT_WINDOW)).astype(int)

print(f"clusters with {MIN_ACCOUNTS}+ accounts on one address+device: "
      f"{clusters.is_multi_account.sum()}")
print(f"...of those, created inside a {TIGHT_WINDOW}-day window: "
      f"{clusters.is_coordinated.sum()}")
print("\nlargest clusters:")
display(clusters[clusters.is_multi_account == 1]
        .sort_values("n_customers", ascending=False)
        [["shipping_address_id", "device_id", "n_customers", "n_orders", "total_value",
          "median_amount", "signup_window_days", "median_account_age", "is_coordinated"]]
        .head(10))

**Push cluster-level flags (coordinated ring, shared device/IP/address) back onto orders.**

In [ ]:
# ---- push the cluster verdict back down onto every order ----------------
ckey = clusters.set_index(["shipping_address_id", "device_id"])
idx = pd.MultiIndex.from_frame(df[["shipping_address_id", "device_id"]])

df["cluster_account_count"] = idx.map(ckey.n_customers)
df["cluster_signup_window_days"] = idx.map(ckey.signup_window_days)
df["shared_entity_flag"] = idx.map(ckey.is_multi_account).fillna(0).astype(int)
df["coordinated_cluster_flag"] = idx.map(ckey.is_coordinated).fillna(0).astype(int)

# device / IP farms: same device or IP, but addresses deliberately different
df["shared_device_flag"] = (df.device_reuse_count >= 5).astype(int)
df["shared_ip_flag"] = (df.ip_reuse_count >= 5).astype(int)
df["shared_address_flag"] = (df.address_reuse_count >= 5).astype(int)

for f in ["shared_entity_flag", "coordinated_cluster_flag", "shared_device_flag",
          "shared_ip_flag", "shared_address_flag"]:
    print(f"{f:26s} fires on {df[f].sum():5,} orders | "
          f"precision {precision_score(y_true, df[f]):.3f} "
          f"recall {recall_score(y_true, df[f]):.3f}")

print("\nNote how much precision the signup-window condition buys: "
      "coordinated_cluster_flag vs shared_entity_flag.\nThat gap is the hostels and "
      "offices being filtered out.")

**Detect daily order-volume spikes using a rolling 7-day baseline.**

In [ ]:
daily = (df.set_index("timestamp").resample("D")
           .agg(orders=("order_id", "size"), revenue=("order_amount", "sum")))
daily["rolling_mean_7d"] = daily.orders.rolling(7, min_periods=3).mean()
daily["rolling_std_7d"] = daily.orders.rolling(7, min_periods=3).std()
daily["z"] = (daily.orders - daily.rolling_mean_7d) / daily.rolling_std_7d
daily["is_spike"] = (daily.z > 2).astype(int)

print(f"days observed : {len(daily)}")
print(f"mean orders/day: {daily.orders.mean():.1f}")
print(f"spike days (>2 sigma above own 7-day baseline): {daily.is_spike.sum()}")
display(daily[daily.is_spike == 1][["orders", "rolling_mean_7d", "z"]].head(10))

**Visualize daily volume, hourly pattern, weekday pattern, and fraud rate by hour.**

In [ ]:
hourly = df.groupby("hour_of_day").size()
dow = df.groupby("day_name").size().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])

fig, axes = plt.subplots(2, 2, figsize=(14, 7))

ax = axes[0, 0]
ax.plot(daily.index, daily.orders, lw=.9, color="#8899aa", label="daily orders")
ax.plot(daily.index, daily.rolling_mean_7d, lw=2, color="#4C78A8", label="7-day rolling mean")
sp = daily[daily.is_spike == 1]
ax.scatter(sp.index, sp.orders, color="crimson", zorder=5, s=28, label="spike (>2 sigma)")
ax.set_title("Daily order volume with rolling baseline")
ax.legend(fontsize=8)

ax = axes[0, 1]
ax.bar(hourly.index, hourly.values, color="#4C78A8")
ax.bar([h for h in hourly.index if 1 <= h < 5],
       [hourly[h] for h in hourly.index if 1 <= h < 5], color="crimson")
ax.set_title("Orders by hour of day (odd hours in red)")
ax.set_xlabel("hour")

ax = axes[1, 0]
ax.bar(range(7), dow.values, color="#4C78A8")
ax.set_xticks(range(7))
ax.set_xticklabels([d[:3] for d in dow.index])
ax.set_title("Orders by day of week")

ax = axes[1, 1]
fr_by_hour = pd.DataFrame({"hour": df.hour_of_day, "fraud": y_true}).groupby("hour").fraud.mean()
ax.bar(fr_by_hour.index, 100 * fr_by_hour.values, color="crimson")
ax.axhline(100 * y_true.mean(), color="k", ls="--", lw=1, label="overall fraud rate")
ax.set_title("Fraud rate by hour (%) — measurement only")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Create spike-day, velocity, and odd-hour-new-account flags.**

In [ ]:
# push the day-level verdict back onto each order
spike_days = set(daily[daily.is_spike == 1].index.normalize())
df["is_spike_day"] = df.order_date.isin(spike_days).astype(int)

# a customer-level burst matters more than a marketplace-wide one
df["velocity_flag"] = (df.orders_last_24h >= 3).astype(int)
df["odd_hour_new_account_flag"] = ((df.is_odd_hour == 1) &
                                   (df.account_age_days <= 14)).astype(int)

for f in ["is_spike_day", "velocity_flag", "odd_hour_new_account_flag"]:
    print(f"{f:28s} fires on {df[f].sum():5,} orders | "
          f"precision {precision_score(y_true, df[f]):.3f} "
          f"recall {recall_score(y_true, df[f]):.3f}")

**Combine all flags into a weighted risk score and Low/Medium/High bucket.**

In [ ]:
# strong composite rules (each mirrors a real fraud archetype)
df["new_account_high_value_flag"] = ((df.account_age_days <= 3) &
                                     (df.amount_vs_category_avg >= 2.0)).astype(int)
# Return abuse needs a third condition, and picking the right one matters a lot.
# "High return rate + several orders" alone also describes a genuine clothing shopper who
# orders three sizes and sends two back - that rule scores ~0.40 precision. Gating on the
# customer's AVERAGE BASKET instead lifts it to ~0.73: refund abusers cycle expensive items,
# bracketing shoppers cycle cheap ones. Same recall, far fewer real customers held.
df["return_abuse_flag"] = ((df.customer_return_rate >= 0.5) &
                           (df.customer_order_count >= 5) &
                           (df.customer_avg_amount >= 8000)).astype(int)

# Points are on a 0-100 scale directly and the total is capped at 100 -- NOT divided by
# the sum of all weights. Dividing would mean a single strong signal could never on its own
# reach a high score, because no order ever trips every rule at once. Capping keeps the
# points readable: "coordinated cluster = 45 points" means the same thing in every context.
RISK_WEIGHTS = {
    "coordinated_cluster_flag":     45,   # ring: many accounts, one roof/device, tight signups
    "new_account_high_value_flag":  35,   # brand-new account buying big
    "return_abuse_flag":            35,   # order-and-return cycling
    "shared_device_flag":           20,   # device farm
    "odd_hour_new_account_flag":    18,   # 1-5 AM from a fresh account
    "shared_address_flag":          15,   # drop address
    "isolation_forest_flag":        15,   # multivariate anomaly
    "shared_ip_flag":               12,   # IP farm
    "velocity_flag":                10,   # 3+ orders in 24h
    "billing_mismatch":              8,   # billing != shipping
    "iqr_flag":                      8,   # oversized basket for its category
}

SCORE_CAP = 100
df["risk_score_raw"] = sum(df[f] * w for f, w in RISK_WEIGHTS.items())
df["risk_score"] = df.risk_score_raw.clip(upper=SCORE_CAP).astype(float)

LOW_MED, MED_HIGH = 20, 40          # justified by the sweep in the next cell
df["risk_bucket"] = pd.Categorical(
    np.select([df.risk_score >= MED_HIGH, df.risk_score >= LOW_MED],
              ["High", "Medium"], default="Low"),
    categories=["Low", "Medium", "High"], ordered=True)

print("weights (points):")
for f, w in RISK_WEIGHTS.items():
    print(f"  {f:30s} {w:3d}   fires on {df[f].sum():5,} orders")
print(f"\nrisk_score: mean {df.risk_score.mean():.1f} | median "
      f"{df.risk_score.median():.1f} | max {df.risk_score.max():.1f}")
print("\nbucket distribution:")
display(pd.DataFrame({
    "orders": df.risk_bucket.value_counts().reindex(["Low", "Medium", "High"]),
    "pct_of_orders": (100 * df.risk_bucket.value_counts(normalize=True)
                      .reindex(["Low", "Medium", "High"])).round(2)}))

**Sweep the risk-score threshold to see the precision/recall trade-off.**

In [ ]:
y_true = true_labels()

# Where should the High threshold sit? Sweep it and look at the trade-off.
sweep = []
for thr in range(5, 101, 5):
    pred = (df.risk_score >= thr).astype(int)
    if pred.sum() == 0:
        continue
    sweep.append({"threshold": thr, "flagged": int(pred.sum()),
                  "precision": precision_score(y_true, pred),
                  "recall": recall_score(y_true, pred),
                  "f1": f1_score(y_true, pred)})
sweep = pd.DataFrame(sweep).set_index("threshold")
display(sweep)

best = sweep.f1.idxmax()
chosen = sweep.loc[MED_HIGH]
print(f"F1 peaks at threshold {best} (F1 = {sweep.loc[best, 'f1']:.3f}).")
print(f"High is set at {MED_HIGH}: {int(chosen.flagged):,} orders flagged "
      f"({100 * chosen.flagged / len(df):.1f}% of volume), precision "
      f"{chosen.precision:.3f}, recall {chosen.recall:.3f}, F1 {chosen.f1:.3f}.")
print("\nWhy not just take the F1 peak? F1 weights precision and recall equally, and a "
      "fraud team\ndoes not. Every false positive is a real customer whose order got held. "
      "The threshold is\nreally a capacity decision: pick the row whose 'flagged' count "
      "matches what the review\nteam can work in a day, then report the precision and "
      "recall that comes with it.")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sweep.index, sweep.precision, marker="o", label="precision")
ax.plot(sweep.index, sweep.recall, marker="o", label="recall")
ax.plot(sweep.index, sweep.f1, marker="o", label="F1")
ax.axvline(MED_HIGH, color="crimson", ls="--", label=f"chosen High cut = {MED_HIGH}")
ax.set_xlabel("risk_score threshold")
ax.set_title("Precision / recall trade-off across risk thresholds")
ax.legend()
plt.tight_layout()
plt.show()

**Build the confusion matrix at the chosen threshold.**

In [ ]:
y_pred = (df.risk_bucket == "High").astype(int).values

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("CONFUSION MATRIX  (rows = actual, columns = predicted)")
display(pd.DataFrame(cm,
                     index=["actual legit", "actual FRAUD"],
                     columns=["predicted legit", "predicted FRAUD"]))

print(f"True Negatives  {tn:5,}  legit orders correctly let through")
print(f"False Positives {fp:5,}  genuine customers wrongly held  <- analyst cost")
print(f"False Negatives {fn:5,}  fraud we shipped                <- chargeback cost")
print(f"True Positives  {tp:5,}  fraud caught")

**Print the full classification report (precision, recall, F1, ROC-AUC, PR-AUC).**

In [ ]:
print("CLASSIFICATION REPORT")
print(classification_report(y_true, y_pred, target_names=["legit", "FRAUD"], digits=3))

prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
pr_auc = average_precision_score(y_true, df.risk_score)
roc = roc_auc_score(y_true, df.risk_score)

print(f"Precision (fraud) : {prec:.3f}   of orders we flag, {100 * prec:.0f}% really are fraud")
print(f"Recall    (fraud) : {rec:.3f}   we catch {100 * rec:.0f}% of all fraud")
print(f"F1        (fraud) : {f1:.3f}")
print(f"PR-AUC            : {pr_auc:.3f}   (random baseline = {y_true.mean():.3f})")
print(f"ROC-AUC           : {roc:.3f}")
print(f"\nAccuracy would be {(tp + tn) / len(df):.3f} — but predicting 'never fraud' "
      f"scores {1 - y_true.mean():.3f}.\nThat is why accuracy is not reported as a headline "
      "number.")

**Plot the precision-recall curve and check that risk buckets rank fraud correctly.**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

p, r, _ = precision_recall_curve(y_true, df.risk_score)
axes[0].plot(r, p, lw=2)
axes[0].axhline(y_true.mean(), ls="--", color="grey", label="random baseline")
axes[0].scatter([rec], [prec], color="crimson", zorder=5, s=60,
                label=f"chosen cut ({MED_HIGH})")
axes[0].set_xlabel("recall")
axes[0].set_ylabel("precision")
axes[0].set_title(f"Precision-Recall curve (PR-AUC = {pr_auc:.3f})")
axes[0].legend()

bucket_rate = pd.DataFrame({"bucket": df.risk_bucket, "fraud": y_true})
summary = (bucket_rate.groupby("bucket", observed=True)
           .agg(orders=("fraud", "size"), fraud=("fraud", "sum")))
summary["fraud_rate_pct"] = (100 * summary.fraud / summary.orders).round(2)
summary["pct_of_all_fraud"] = (100 * summary.fraud / summary.fraud.sum()).round(2)
axes[1].bar(summary.index.astype(str), summary.fraud_rate_pct,
            color=["#4C78A8", "orange", "crimson"])
axes[1].axhline(100 * y_true.mean(), ls="--", color="k", lw=1, label="base rate")
axes[1].set_ylabel("fraud rate within bucket (%)")
axes[1].set_title("Does the bucket mean anything?")
axes[1].legend()
plt.tight_layout()
plt.show()

display(summary)
print("A working risk score has a steeply rising fraud rate across buckets. "
      "Flat buckets = the score isn't ranking.")

**Error analysis: recall by fraud pattern, and what genuine orders got wrongly flagged.**

In [ ]:
# Which fraud did we miss, and what did we wrongly flag? Error analysis is the part
# that turns a portfolio project into an interview conversation.
audit = df[["order_id", "risk_score", "risk_bucket"]].copy()
audit["is_fraud"] = y_true
audit["fraud_pattern"] = true_patterns()

print("RECALL BY FRAUD ARCHETYPE — what the system is blind to:")
caught = (audit[audit.is_fraud == 1]
          .assign(caught=lambda d: (d.risk_bucket == "High").astype(int))
          .groupby("fraud_pattern")
          .agg(fraud_orders=("caught", "size"), caught=("caught", "sum")))
caught["recall_pct"] = (100 * caught.caught / caught.fraud_orders).round(1)
display(caught.sort_values("recall_pct", ascending=False))

print("\nFALSE POSITIVES — genuine orders we held, and why they looked bad:")
fp_rows = df[(y_pred == 1) & (y_true == 0)]
if len(fp_rows):
    reasons = {f: int(fp_rows[f].sum()) for f in RISK_WEIGHTS}
    display(pd.Series(reasons).sort_values(ascending=False).to_frame("false positives triggering it"))
    display(fp_rows[["order_id", "account_age_days", "order_amount",
                     "amount_vs_category_avg", "device_reuse_count",
                     "customer_return_rate", "risk_score"]].head(5))
notes = [
    "",
    "Read the recall table as a map of blind spots, not as a scoreboard:",
    "",
    "  ring / farm / new-account   caught, because they leave entity-graph fingerprints:",
    "                              one device or address behind many accounts is visible even",
    "                              when each individual order looks perfectly ordinary.",
    "",
    "  return_abuse                partly caught, on purpose. The flag is gated on the customer's",
    "                              average basket so that genuine 'order three sizes, return two'",
    "                              shoppers are not held. Loosening it would lift recall and hold",
    "                              a lot of real clothing customers - a bad trade.",
    "",
    "  account_takeover            almost entirely missed, and honestly so. An old trusted account",
    "                              with a normal-sized basket looks fine on every single-row",
    "                              feature. Catching it needs per-customer baselines - usual device,",
    "                              usual address, usual basket - and a deviation-from-self score.",
    "                              That is the natural next iteration of this project.",
    "",
    "  mislabelled_legit           0% recall is the CORRECT result. Those rows are label noise:",
    "                              ordinary orders carrying a bad chargeback. A system that",
    "                              'caught' them would be fitting the noise, not the fraud.",
]
print("\n".join(notes))

**Save the final scored dataset to CSV.**

In [ ]:
FLAG_COLS = [*RISK_WEIGHTS.keys(), "iqr_flag_global", "iqr_flag_category",
             "iqr_flag_deviation", "shared_entity_flag", "is_spike_day",
             "anomaly_score", "risk_score_raw", "risk_score", "risk_bucket"]

output = df.copy()
output["is_fraud"] = true_labels()               # kept for offline evaluation
output["fraud_pattern"] = true_patterns()

ORIGINAL = ["order_id", "timestamp", "customer_id", "account_creation_date", "order_amount",
            "product_category", "quantity", "payment_method", "shipping_address_id",
            "billing_address_id", "device_id", "ip_address", "is_returned",
            "return_reason", "return_timestamp"]
COLS = (ORIGINAL + [c for c in ENGINEERED if c not in ORIGINAL]
        + [c for c in FLAG_COLS if c not in ORIGINAL] + ["is_fraud", "fraud_pattern"])
COLS = list(dict.fromkeys([c for c in COLS if c in output.columns]))

output = output[COLS].sort_values("timestamp").reset_index(drop=True)
SCORED_PATH = OUT_DIR / "ecommerce_orders_scored.csv"
output.to_csv(SCORED_PATH, index=False)

print(f"saved -> {SCORED_PATH}")
print(f"        {len(output):,} rows x {len(output.columns)} columns")
print("\nhighest-risk orders:")
display(output.nlargest(5, "risk_score")[
    ["order_id", "customer_id", "order_amount", "account_age_days",
     "device_reuse_count", "risk_score", "risk_bucket", "is_fraud"]])

**Bundle the trained model with feature list, stats, and weights, and save it.**

In [ ]:
from datetime import datetime

cat_stats = (df.groupby("product_category")
               .agg(mean_amount=("order_amount", "mean"),
                    std_amount=("order_amount", "std"))
               .to_dict("index"))

model_bundle = {
    "model": iso,
    "model_type": "IsolationForest",
    "feature_names": IF_FEATURES,                  # order matters
    "contamination": CONTAMINATION,
    "category_stats": cat_stats,                   # for order_amount_deviation
    "risk_weights": RISK_WEIGHTS,
    "risk_score_cap": SCORE_CAP,
    "bucket_thresholds": {"low_medium": LOW_MED, "medium_high": MED_HIGH},
    "iqr_bounds": {"deviation_upper": float(d_hi)},
    "cluster_rules": {"min_accounts": MIN_ACCOUNTS, "tight_signup_days": TIGHT_WINDOW},
    "trained_on_rows": int(len(df)),
    "trained_at": datetime.now().isoformat(timespec="seconds"),
    "sklearn_version": __import__("sklearn").__version__,
    "version": "1.0.0",
}

MODEL_PATH = OUT_DIR / "fraud_isolation_forest.joblib"
joblib.dump(model_bundle, MODEL_PATH, compress=3)
print(f"saved -> {MODEL_PATH}  ({MODEL_PATH.stat().st_size / 1024:.0f} KB)")
print("bundle keys:", list(model_bundle.keys()))

**Reload the saved model and confirm its predictions match the original.**

In [ ]:
loaded = joblib.load(MODEL_PATH)
check = -loaded["model"].score_samples(df[loaded["feature_names"]].astype(float).values)

print("reloaded model:", loaded["model_type"], "v" + loaded["version"],
      "| trained", loaded["trained_at"])
print("scores identical to in-memory model:",
      bool(np.allclose(check, df.anomaly_score.values)))
print("max difference:", float(np.abs(check - df.anomaly_score.values).max()))

**Define and test the single-order scoring function used by the backend API.**

In [ ]:
def score_one_order(order: dict, bundle: dict, history_lookups: dict) -> dict:
    """Score a single incoming order.

    order            : raw order fields (amount, category, timestamps, ids...)
    bundle           : the joblib bundle loaded above
    history_lookups  : counts the API must fetch from the DB, e.g.
                       {"device_reuse_count": 7, "address_reuse_count": 2,
                        "orders_last_24h": 1, "customer_return_rate": 0.0,
                        "ip_reuse_count": 1, "address_device_pair_count": 7,
                        "customer_order_count": 3, "customer_avg_amount": 2400,
                        "cluster_signup_window_days": 4}
    """
    cat = bundle["category_stats"].get(order["product_category"])
    if cat is None:                       # unseen category -> fall back to neutral
        cat = {"mean_amount": order["order_amount"], "std_amount": 1.0}

    ts = pd.Timestamp(order["timestamp"])
    deviation = (order["order_amount"] - cat["mean_amount"]) / (cat["std_amount"] or 1.0)
    amount_ratio = order["order_amount"] / (cat["mean_amount"] or 1.0)
    account_age = (ts.normalize() - pd.Timestamp(order["account_creation_date"])).days
    is_odd_hour = int(1 <= ts.hour < 5)

    feat = {
        "account_age_days": account_age,
        "log_order_amount": np.log1p(order["order_amount"]),
        "order_amount_deviation": deviation,
        "quantity": order.get("quantity", 1),
        "orders_last_24h": history_lookups.get("orders_last_24h", 0),
        "customer_return_rate": history_lookups.get("customer_return_rate", 0.0),
        "address_reuse_count": history_lookups.get("address_reuse_count", 1),
        "device_reuse_count": history_lookups.get("device_reuse_count", 1),
        "ip_reuse_count": history_lookups.get("ip_reuse_count", 1),
        "address_device_pair_count": history_lookups.get("address_device_pair_count", 1),
        "is_odd_hour": is_odd_hour,
        "hour_of_day": ts.hour,
        "day_of_week": ts.dayofweek,
        "billing_mismatch": int(order["billing_address_id"] != order["shipping_address_id"]),
        "customer_order_count": history_lookups.get("customer_order_count", 1),
    }
    X1 = np.array([[feat[f] for f in bundle["feature_names"]]], dtype=float)
    anomaly = float(-bundle["model"].score_samples(X1)[0])
    iso_flag = int(bundle["model"].predict(X1)[0] == -1)

    w = bundle["risk_weights"]
    window = history_lookups.get("cluster_signup_window_days", 9_999)
    rules = bundle["cluster_rules"]
    fired = {
        "coordinated_cluster_flag": int(feat["address_device_pair_count"] >= rules["min_accounts"]
                                        and window <= rules["tight_signup_days"]),
        "new_account_high_value_flag": int(account_age <= 3 and amount_ratio >= 2.0),
        "shared_device_flag": int(feat["device_reuse_count"] >= 5),
        "return_abuse_flag": int(feat["customer_return_rate"] >= .5
                                 and feat["customer_order_count"] >= 5
                                 and history_lookups.get("customer_avg_amount", 0) >= 8000),
        "shared_address_flag": int(feat["address_reuse_count"] >= 5),
        "odd_hour_new_account_flag": int(is_odd_hour and account_age <= 14),
        "isolation_forest_flag": iso_flag,
        "velocity_flag": int(feat["orders_last_24h"] >= 3),
        "shared_ip_flag": int(feat["ip_reuse_count"] >= 5),
        "billing_mismatch": feat["billing_mismatch"],
        "iqr_flag": int(deviation > bundle["iqr_bounds"]["deviation_upper"]),
    }
    raw = sum(w[k] * v for k, v in fired.items())
    score = float(min(raw, bundle["risk_score_cap"]))
    cuts = bundle["bucket_thresholds"]
    bucket = ("High" if score >= cuts["medium_high"]
              else "Medium" if score >= cuts["low_medium"] else "Low")
    return {"risk_score": score, "risk_bucket": bucket, "anomaly_score": anomaly,
            "triggered": [k for k, v in fired.items() if v]}


# smoke-test it on a real row so the function is proven, not just written
sample = df.nlargest(1, "risk_score").iloc[0]
demo = score_one_order(
    order={"timestamp": sample.timestamp, "account_creation_date": sample.account_creation_date,
           "order_amount": sample.order_amount, "product_category": sample.product_category,
           "quantity": sample.quantity, "billing_address_id": sample.billing_address_id,
           "shipping_address_id": sample.shipping_address_id},
    bundle=loaded,
    history_lookups={"orders_last_24h": int(sample.orders_last_24h),
                     "customer_return_rate": float(sample.customer_return_rate),
                     "address_reuse_count": int(sample.address_reuse_count),
                     "device_reuse_count": int(sample.device_reuse_count),
                     "ip_reuse_count": int(sample.ip_reuse_count),
                     "address_device_pair_count": int(sample.address_device_pair_count),
                     "customer_order_count": int(sample.customer_order_count),
                     "customer_avg_amount": float(sample.customer_avg_amount),
                     "cluster_signup_window_days": float(sample.cluster_signup_window_days)})

print("backend scoring of order", sample.order_id)
for k, v in demo.items():
    print(f"  {k:15s} {v}")
print(f"\nnotebook said: risk_score {sample.risk_score}, bucket {sample.risk_bucket}")